# tttLRM: Test-Time Training for 3D Reconstruction

Interactive single-GPU inference notebook. Directly loads the model in Python (no `torchrun` / DDP needed).

**Requirements:** GPU with >= 24GB VRAM (A100 recommended). `Runtime > Change runtime type > GPU`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Setup

In [ ]:
%%bash
if [ ! -d "tttLRM" ]; then
    git clone https://github.com/chenwang/tttLRM.git
fi

In [ ]:
import os
os.chdir('tttLRM')
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0;7.5;8.0;8.6;8.9;9.0'

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118 xformers
!pip install -q -U setuptools wheel packaging ninja
!pip install -q flash_attn==2.5.9.post1 --no-build-isolation
!pip install -q -r requirements.txt
!pip install -q plotly

In [ ]:
# Download checkpoints
!huggingface-cli download chenwang/tttLRM --local-dir checkpoints --local-dir-use-symlinks False
!mkdir -p depth_anything_v2 && wget -q 'https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth?download=true' -O depth_anything_v2/depth_anything_vits.pth
!ls -lh checkpoints/

## 2. Patch DDP for Single-GPU

The model code uses `torch.distributed` and sequence parallelism throughout. We monkey-patch `utils.sp_support` so all distributed ops become identity (no-ops on a single GPU). This avoids needing `torchrun`.

In [ ]:
import sys
import types
import torch

# Create a fake sp_support module that makes all distributed ops into identity
fake_sp = types.ModuleType('utils.sp_support')

def _identity(x, *args, **kwargs):
    """Identity: for single GPU, broadcast/scatter/gather are all no-ops."""
    if kwargs.get('different_size', False):
        return x, x.shape
    return x

fake_sp.init_sp_group = lambda sp_size: None
fake_sp.is_sp = lambda: False
fake_sp.get_sp_rank = lambda: 0
fake_sp.get_sp_world_size = lambda: 1
fake_sp.get_sp_replicas = lambda: 1
fake_sp.get_sp_replica_id = lambda: 0
fake_sp.sp_all_reduce = lambda tensor, op=None: tensor
fake_sp.sp_all_gather = _identity
fake_sp.sp_input_broadcast_scatter = _identity
fake_sp.sp_broadcast = _identity
fake_sp.sp_local_scatter = _identity
fake_sp.sp_gather_scatter = _identity

# Inject into sys.modules BEFORE any model imports
sys.modules['utils.sp_support'] = fake_sp
sys.modules['utils'] = __import__('utils')  # keep the real utils package
sys.modules['utils'].sp_support = fake_sp

print("Patched sp_support for single-GPU inference")

## 3. Prepare Data

**Option A:** Download a DL3DV benchmark scene  
**Option B:** Upload your own scene (zip with `opencv_cameras.json` + `images_undistort/`)

In [ ]:
DATA_SOURCE = "dl3dv"  # @param ["dl3dv", "custom_upload"]
SCENE_HASH = "032dee9fb0a8bc1b90871dc5fe950080d0bcd3caf166447f44e60ca50ac04ec7"  # @param {type:"string"}

In [ ]:
import json
from pathlib import Path

if DATA_SOURCE == "dl3dv":
    !python data/dl3dv_eval_download.py \
        --odir ./data_example/dl3dv_benchmark \
        --subset hash --only_level4 --hash {SCENE_HASH}

    sys.path.insert(0, '.')
    from data.dl3dv_format_convert import process_one_scene

    benchmark_dir = Path('./data_example/dl3dv_benchmark')
    output_base = Path('./data_example/dl3dv_processed')
    output_base.mkdir(parents=True, exist_ok=True)

    data_paths = []
    for scene_dir in sorted(benchmark_dir.iterdir()):
        if scene_dir.is_dir() and not scene_dir.name.startswith('.'):
            scene_output = output_base / scene_dir.name
            scene_output.mkdir(parents=True, exist_ok=True)
            if process_one_scene(scene_dir, scene_output):
                data_paths.append(str((scene_output / 'opencv_cameras.json').resolve()))

    data_path_json = './data_example/dl3dv_sample_data_path.json'
    with open(data_path_json, 'w') as f:
        json.dump(data_paths, f, indent=2)
    print(f"Prepared {len(data_paths)} scene(s)")

elif DATA_SOURCE == "custom_upload":
    from google.colab import files
    print("Upload a zip containing: your_scene/opencv_cameras.json + your_scene/images_undistort/")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !mkdir -p ./data_example/custom && unzip -o "{zip_name}" -d ./data_example/custom/
    cam_files = list(Path('./data_example/custom').rglob('opencv_cameras.json'))
    assert cam_files, "No opencv_cameras.json found!"
    data_paths = [str(f.resolve()) for f in cam_files]
    data_path_json = './data_example/custom_data_path.json'
    with open(data_path_json, 'w') as f:
        json.dump(data_paths, f, indent=2)
    print(f"Found {len(data_paths)} scene(s)")

In [ ]:
# Preview input images
import matplotlib.pyplot as plt
from PIL import Image

with open(data_paths[0]) as f:
    scene_data = json.load(f)
scene_dir = str(Path(data_paths[0]).parent)
frames = scene_data['frames']

n_preview = min(8, len(frames))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < n_preview:
        ax.imshow(Image.open(os.path.join(scene_dir, frames[i]['file_path'])))
        ax.set_title(f"Frame {i}")
    ax.axis('off')
plt.suptitle(f"{scene_data.get('scene_name', 'scene')} — {len(frames)} frames", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Load Model & Run Inference (Pure Python, No DDP)

In [ ]:
import numpy as np
import random
import omegaconf
from easydict import EasyDict as edict

torch.manual_seed(777)
np.random.seed(777)
random.seed(777)
device = "cuda:0"

# Load config
CKPT_PATH = "./checkpoints/dl3dv_full.pt"
CONFIG_PATH = "./configs/dl3dv_full.yaml"
NUM_VIEWS = 16  # @param [16] {type:"integer"}

config = omegaconf.OmegaConf.load(CONFIG_PATH)
config = edict(omegaconf.OmegaConf.to_container(config, resolve=True))

# Override for single-GPU evaluation
config.evaluation = True
config.inference = False
config.sp_size = 1
config.training.target_has_input = False
config.training.batch_size_per_gpu = 1
config.training.num_input_views = NUM_VIEWS
config.training.num_target_views = NUM_VIEWS
config.training.num_virtual_views = NUM_VIEWS
config.training.num_views = 35
config.training.data_repeat = 1
config.training.perceptual_loss_weight = 0.0
config.training.reset_training_state = False
config.training.torch_compile = False  # avoid compile overhead in notebook
config.training.view_selector = edict({"type": "kmeans"})
config.training.frame_method = "mean_cam"
config.training.dataset_path = data_path_json
config.model.use_anything = False
config.model.act_ckpt = False
config.model.gaussians.usage_threshold = 0.001
config.kmeans_input = True
config.metrics_only = False
config.num_frames = 8
config.eval_dataset_path = data_path_json

print(f"Config loaded: {CONFIG_PATH}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Input views: {NUM_VIEWS}")

In [ ]:
import importlib
import copy

# Load dataset
dataset_name = config.training.get("dataset_name", "data.dataset_scene.Dataset")
module_name, class_name = dataset_name.rsplit(".", 1)
Dataset = importlib.import_module(module_name).__dict__[class_name]

eval_config = copy.deepcopy(config)
eval_config.training.dataset_path = config.eval_dataset_path
eval_config.evaluation = True
eval_config.training.sample_ar = False
eval_config.training.sample_mixed_length = False
eval_config.training.data_repeat = 1
eval_dataset = Dataset(eval_config)
print(f"Dataset loaded: {len(eval_dataset)} scene(s)")

In [ ]:
# Load model
module_name, class_name = config.model.class_name.rsplit(".", 1)
tttLRM_cls = importlib.import_module(module_name).__dict__[class_name]
model = tttLRM_cls(config).to(device)

# Load checkpoint
checkpoint = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(checkpoint['model'], strict=False)
model.eval()

overview = model.get_overview()
total_params = sum(v for v in overview.values())
print(f"Model loaded: {total_params / 1e6:.1f}M trainable parameters")
print(f"  tokenizer: {overview.tokenizer / 1e6:.1f}M")
print(f"  blocks:    {overview.blocks / 1e6:.1f}M")
print(f"  decoder:   {overview.image_token_decoder / 1e6:.1f}M")

In [ ]:
from torch.utils.data import DataLoader

eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

OUT_DIR = "./evaluation/colab_eval"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Running inference on {len(eval_dataset)} scene(s)...")
print(f"Output directory: {OUT_DIR}")

In [ ]:
import time

all_results = []

with torch.no_grad(), torch.autocast(enabled=config.training.use_amp, device_type="cuda", dtype=torch.bfloat16):
    for i, batch in enumerate(eval_dataloader):
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        t0 = time.time()
        result = model(batch)
        t1 = time.time()

        # Save evaluations (images, PLY, metrics, video)
        model.save_evaluations(OUT_DIR, result, batch)

        psnr = result.loss_metrics.psnr_real.item()
        print(f"Scene {i}: PSNR={psnr:.2f} dB, time={t1 - t0:.1f}s")

        all_results.append(result)
        torch.cuda.empty_cache()

print("\nInference complete!")

## 5. Evaluation Metrics

In [ ]:
import pandas as pd
import glob

eval_dir = Path(OUT_DIR)
csv_files = sorted(glob.glob(str(eval_dir / '**' / 'metrics.csv'), recursive=True))

for csv_file in csv_files:
    scene_name = Path(csv_file).parent.name
    df = pd.read_csv(csv_file, skipinitialspace=True)
    mean_row = df[df['index'] == 'mean']
    if not mean_row.empty:
        print(f"Scene: {scene_name}")
        print(f"  PSNR:  {float(mean_row['psnr'].values[0]):.2f}")
        print(f"  SSIM:  {float(mean_row['ssim'].values[0]):.4f}")
        print(f"  LPIPS: {float(mean_row['lpips'].values[0]):.4f}")

## 6. Side-by-Side: Ground Truth vs. Rendering

In [ ]:
scene_dirs = sorted([d for d in eval_dir.iterdir() if d.is_dir()])
if not scene_dirs:
    print("No output found.")
else:
    scene_out = scene_dirs[0]
    target_dir = scene_out / 'target'
    render_dir = scene_out / 'rendering'

    if target_dir.exists() and render_dir.exists():
        target_imgs = sorted(target_dir.glob('*.png'))
        n_show = min(6, len(target_imgs))

        fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
        if n_show == 1:
            axes = axes.reshape(2, 1)
        for i in range(n_show):
            gt = Image.open(target_dir / f'{i}.png')
            rd = Image.open(render_dir / f'{i}.png')
            axes[0, i].imshow(gt)
            axes[0, i].set_title(f'GT #{i}')
            axes[0, i].axis('off')
            axes[1, i].imshow(rd)
            axes[1, i].set_title(f'Rendered #{i}')
            axes[1, i].axis('off')
        fig.suptitle('Ground Truth (top) vs. tttLRM Rendering (bottom)', fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("No images saved. Check metrics_only setting.")

## 7. Interactive 3D Gaussian Visualization

In [ ]:
ply_files = sorted(glob.glob(str(eval_dir / '**' / '*.ply'), recursive=True))

if ply_files:
    from plyfile import PlyData
    import plotly.graph_objects as go

    ply = PlyData.read(ply_files[0])
    vertex = ply['vertex']
    xyz = np.stack([vertex['x'], vertex['y'], vertex['z']], axis=-1)
    # SH DC -> approx RGB
    r = np.clip(vertex['f_dc_0'] * 0.2824 + 0.5, 0, 1)
    g = np.clip(vertex['f_dc_1'] * 0.2824 + 0.5, 0, 1)
    b = np.clip(vertex['f_dc_2'] * 0.2824 + 0.5, 0, 1)

    print(f"Loaded {len(xyz)} Gaussians from {ply_files[0]}")

    # Subsample for visualization
    max_pts = 100_000
    if len(xyz) > max_pts:
        idx = np.random.choice(len(xyz), max_pts, replace=False)
        xyz_v, r_v, g_v, b_v = xyz[idx], r[idx], g[idx], b[idx]
        print(f"Subsampled to {max_pts} points")
    else:
        xyz_v, r_v, g_v, b_v = xyz, r, g, b

    colors = [f'rgb({int(ri*255)},{int(gi*255)},{int(bi*255)})' for ri, gi, bi in zip(r_v, g_v, b_v)]

    fig = go.Figure(data=[go.Scatter3d(
        x=xyz_v[:, 0], y=xyz_v[:, 1], z=xyz_v[:, 2],
        mode='markers',
        marker=dict(size=1.5, color=colors, opacity=0.8)
    )])
    fig.update_layout(
        title=f'3D Gaussians ({len(xyz)} total, showing {len(xyz_v)})',
        scene=dict(aspectmode='data'),
        width=800, height=600, margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()
else:
    print("No PLY files. Set metrics_only=False.")

## 8. Render Custom Novel Views

Use the predicted Gaussians to render from arbitrary camera poses — no need to re-run the full model.

In [ ]:
from model.gaussian_renderer import GaussianRenderer

result = all_results[0]
gaussians = result.gaussians
H, W = result.input.image.shape[-2:]

# Get input cameras for reference
input_c2ws = result.input.c2w[0]   # (V, 4, 4)
input_intr = result.input.fxfycxcy[0]  # (V, 4)

def render_from_camera(gaussians, c2w, fxfycxcy, H, W, sh_degree=3):
    """Render a single view from the predicted Gaussians."""
    with torch.no_grad(), torch.autocast(enabled=False, device_type="cuda"):
        rendering = GaussianRenderer.render(
            gaussians['xyz'][0], gaussians['feature'][0],
            gaussians['scale'][0], gaussians['rotation'][0],
            gaussians['opacity'][0],
            c2w.float(), fxfycxcy.float(), W, H,
            sh_degree, 0.1, 1e10
        )
    img = rendering.squeeze(0).clamp(0, 1).cpu().numpy()  # (H, W, 3+D)
    return img[:, :, :3]

# Render from each input camera to verify
n_render = min(4, input_c2ws.shape[0])
fig, axes = plt.subplots(1, n_render, figsize=(4 * n_render, 4))
if n_render == 1:
    axes = [axes]
for i in range(n_render):
    img = render_from_camera(gaussians, input_c2ws[i], input_intr[i], H, W)
    axes[i].imshow(img)
    axes[i].set_title(f'View {i}')
    axes[i].axis('off')
plt.suptitle('Re-rendered from input cameras (using predicted Gaussians)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Render an interpolated camera trajectory and display as video
from utils import camera_utils
from IPython.display import HTML
from base64 import b64encode
import cv2
import subprocess

V = input_c2ws.shape[0]
c2ws_cpu = input_c2ws.detach().cpu().float()
intr_cpu = input_intr.detach().cpu().float()

# Build 3x3 intrinsic matrices
intr_mat = torch.zeros((V, 3, 3))
intr_mat[:, 0, 0] = intr_cpu[:, 0]
intr_mat[:, 1, 1] = intr_cpu[:, 1]
intr_mat[:, 0, 2] = intr_cpu[:, 2]
intr_mat[:, 1, 2] = intr_cpu[:, 3]

# Wrap around for smooth loop
c2ws_loop = torch.cat([c2ws_cpu, c2ws_cpu[:1]], dim=0)
intr_loop = torch.cat([intr_mat, intr_mat[:1]], dim=0)

FRAMES_PER_TRANSITION = 8  # @param {type:"slider", min:4, max:16, step:2}
interp_c2ws, interp_intr, _ = camera_utils.get_interpolated_poses_many(
    c2ws_loop[:, :3, :4], intr_loop, steps_per_transition=FRAMES_PER_TRANSITION
)

# Build full 4x4 c2w + fxfycxcy
n_frames = interp_c2ws.shape[0]
full_c2ws = torch.eye(4).unsqueeze(0).repeat(n_frames, 1, 1)
full_c2ws[:, :3, :4] = interp_c2ws
full_c2ws = full_c2ws.to(device)

interp_fxfycxcy = torch.zeros(n_frames, 4)
interp_fxfycxcy[:, 0] = interp_intr[:, 0, 0]
interp_fxfycxcy[:, 1] = interp_intr[:, 1, 1]
interp_fxfycxcy[:, 2] = interp_intr[:, 0, 2]
interp_fxfycxcy[:, 3] = interp_intr[:, 1, 2]
interp_fxfycxcy = interp_fxfycxcy.to(device)

print(f"Rendering {n_frames} frames...")
rendered_frames = []
for i in range(n_frames):
    img = render_from_camera(gaussians, full_c2ws[i], interp_fxfycxcy[i], H, W)
    rendered_frames.append((img * 255).astype(np.uint8))

# Write video
video_path = os.path.join(OUT_DIR, 'novel_view_trajectory.mp4')
tmp_path = video_path.replace('.mp4', '_tmp.mp4')
writer = cv2.VideoWriter(tmp_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (W, H))
for frame in rendered_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()
subprocess.run(f"ffmpeg -y -i {tmp_path} -vcodec libx264 -f mp4 {video_path} -loglevel quiet", shell=True)
if os.path.exists(tmp_path):
    os.remove(tmp_path)

# Display in notebook
with open(video_path, 'rb') as f:
    b64 = b64encode(f.read()).decode()
display(HTML(f'<video width="640" controls autoplay loop><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
print(f"Saved to {video_path}")

## 9. Download Results

In [ ]:
!zip -r /tmp/tttlrm_results.zip {OUT_DIR}

try:
    from google.colab import files
    files.download('/tmp/tttlrm_results.zip')
except ImportError:
    print("Results saved to /tmp/tttlrm_results.zip")
    print("PLY files can be viewed at https://playcanvas.com/supersplat/editor")